# 02 — Text Analytics Pipeline: F0 → F5

**DS 5001 — Exploratory Text Analytics Final Project**

This notebook runs the full text analytics pipeline on the papal encyclicals corpus:

| Stage | Description |
|-------|-------------|
| F0 → F1 | Raw text → paragraphs indexed by document hierarchy |
| F1 → F2 | Tokenization → LIBRARY, TOKEN, VOCAB tables (STADM) |
| F2 → F3 | NLP annotations: POS, lemma, stopwords, sentiment |
| F3 → F4 | TFIDF vectorization |
| F4 → F5 | PCA, LDA topic models, word2vec embeddings |

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from src.pipeline import (
    build_f1_corpus, build_f2_tables, build_f3_annotations,
    build_f4_tfidf, build_f5_models, save_tables,
    PROCESSED_DIR
)

## F0 → F1: Machine Learning Corpus Format

Split raw documents into paragraphs (minimum discursive units),
indexed by document content hierarchy.

In [ ]:
corpus = build_f1_corpus(english_only=True)
print(f"Corpus shape: {corpus.shape}")
print(f"Documents: {corpus['doc_id'].nunique()}")
print(f"Paragraphs: {len(corpus)}")
corpus.head()

## F1 → F2: STADM Tables

Tokenize into LIBRARY (doc metadata), TOKEN (every word), VOCAB (unique terms).

In [ ]:
LIBRARY, TOKEN, VOCAB = build_f2_tables(corpus)
print("LIBRARY:"); display(LIBRARY.head())
print("\nTOKEN:"); display(TOKEN.head(10))
print("\nVOCAB (top 20):"); display(VOCAB.head(20))

## F2 → F3: NLP Annotations

Add POS tags, lemmas, stopword flags, and VADER sentiment scores.

In [ ]:
LIBRARY, TOKEN, VOCAB = build_f3_annotations(TOKEN, VOCAB, LIBRARY)
print("TOKEN with annotations:"); display(TOKEN.head(10))
print("\nVOCAB with annotations:"); display(VOCAB.head(10))
print("\nLIBRARY sentiment:"); display(LIBRARY[['pope','title','sentiment_compound']].head())

## F3 → F4: TFIDF Vectorization

Compute TF-IDF scores and build the document-term matrix.

In [ ]:
LIBRARY, TOKEN, VOCAB, TFIDF_DTM = build_f4_tfidf(TOKEN, VOCAB, LIBRARY)
print(f"Document-term matrix: {TFIDF_DTM.shape}")
print(f"\nTop TFIDF terms per document (first 3 docs):")
for doc_id in TFIDF_DTM.index[:3]:
    top = TFIDF_DTM.loc[doc_id].nlargest(5)
    print(f"  {doc_id[:40]}: {', '.join(top.index)}")

## F4 → F5: Unsupervised Models

Fit PCA, LDA, and word2vec models.

In [ ]:
f5_results = build_f5_models(
    LIBRARY, TOKEN, VOCAB, TFIDF_DTM,
    n_components=10, n_topics=10, w2v_dim=100
)

print("PCA - documents x components:")
display(f5_results['DOC_PCA'].head())
print(f"\nExplained variance: {f5_results['explained_variance'].sum():.1%}")

print("\nLDA - documents x topics:")
display(f5_results['DOC_TOPICS'].head())

print(f"\nword2vec embeddings: {f5_results['EMBEDDINGS'].shape}")

## Save All Tables

In [ ]:
save_tables(LIBRARY, TOKEN, VOCAB, TFIDF_DTM, f5_results)
print("All tables saved!")

# List output files
for f in sorted(PROCESSED_DIR.glob('*.csv')):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name}: {size_kb:.0f} KB")